# Detector de fumaça: do dataset ao modelo

A caixa de fumaça do professor tem 3 sensores e mede **partícula, gás, calor e umidade**, uma
leitura por segundo. Enquanto gravava, o professor apertou o botão BOOT para marcar o que estava
perto da caixa: ar, vapor, chama, fumaça ou álcool.

Neste notebook você abre o dataset, vê a **fusão dos sensores** num gráfico, treina uma **árvore
de decisão** e testa o modelo numa coleta que ele nunca viu.

**Como usar no Colab:**
1. Arquivo → Fazer upload de notebook → escolha este arquivo.
2. Rode as células na ordem (Shift + Enter). Na primeira célula de código, escolha o
   `dataset_detector_fumaca_fiap.csv`.

## 1) O dataset

Cada linha é **um segundo** da caixa.

| Coluna | O que é | Sensor |
|---|---|---|
| `coleta` | quando foi gravada (dia e hora do início) | — |
| `estimulo` | o que estava perto da caixa, marcado no botão BOOT | — |
| `temp` | calor (°C) | BME280 |
| `umid` | umidade (%) | BME280 |
| `raw_h2` | gás: hidrogênio, valor cru. **Cai** quando o gás sobe | SGP30 |
| `raw_ethanol` | gás: etanol, valor cru. **Cai** quando o gás sobe | SGP30 |
| `pm2_5`, `pm10` | massa de partículas (µg/m³) | SPS30 |
| `nc0_5` | contagem de partículas pequenas (por cm³) | SPS30 |
| `incendio` | **o rótulo:** `fire` (fumaça ou chama) ou `not fire` (ar, vapor, álcool) | — |

O professor já preparou o CSV a partir da coleta bruta:
- **saiu o aquecimento** da caixa (os primeiros minutos depois de ligar);
- **saíram os primeiros segundos depois de cada aperto do botão:** a curva só muda 15 a 30 s
  depois. Saem 30 s de cada estímulo e 120 s de cada trecho de ar (a fumaça demora a sair);
- **saíram as colunas que não viram feature:** o contador, a rodada, a pressão e o horário.
  Elas mudam com o momento da gravação, não com o fogo;
- **ficaram 7 sensores:** os que as duas versões do programa da caixa medem do mesmo jeito.

In [ ]:
import os
import pandas as pd

ARQUIVO = "dataset_detector_fumaca_fiap.csv"

if not os.path.exists(ARQUIVO):
    from google.colab import files
    print("Nao achei", ARQUIVO, "- escolha o CSV no seu computador:")
    enviados = files.upload()
    ARQUIVO = list(enviados)[0]

df = pd.read_csv(ARQUIVO)
print(len(df), "linhas")
df.head()

## 2) Quanto tem de cada

Quantos segundos de cada estímulo, em cada coleta, e quanto é fogo.

In [ ]:
display(pd.crosstab(df["coleta"], df["estimulo"]))
df["incendio"].value_counts()

Repare: há muito mais segundos de `not fire` do que de `fire`. Um detector passa quase o tempo
todo vendo ar normal.

## 3) A fusão no gráfico

Cada linha do dataset é um ponto, na cor do estímulo (as cores do LED da caixa). Com **dois
sensores juntos**, cada estímulo ocupa o seu canto. Olhando um eixo só, as cores se misturam:
um sensor sozinho engana.

In [ ]:
import matplotlib.pyplot as plt

CORES = {"ar": "#22c55e", "álcool": "#eab308", "vapor": "#a855f7", "fumaça": "#ef4444", "chama": "#3b82f6"}

fig, axs = plt.subplots(1, 2, figsize=(13, 5))
for estimulo, cor in CORES.items():
    g = df[df["estimulo"] == estimulo]
    axs[0].scatter(g["nc0_5"], g["umid"], s=8, color=cor, alpha=0.6, label=estimulo)
    axs[1].scatter(g["raw_ethanol"], g["temp"], s=8, color=cor, alpha=0.6, label=estimulo)

axs[0].set_xscale("log")   # a fumaca vai a milhares; o ar fica perto de 100
axs[0].set_xlabel("partícula pequena, nc0_5 (por cm³, escala log)")
axs[0].set_ylabel("umidade (%)")
axs[1].set_xlabel("gás etanol, raw_ethanol (valor cru: cai quando o gás sobe)")
axs[1].set_ylabel("calor (°C)")
axs[0].legend()
plt.tight_layout()
plt.show()

- **Partícula × umidade:** a fumaça tem muita partícula; o vapor também tem partícula, mas sobe
  a umidade junto.
- **Gás × calor:** o álcool derruba o valor cru do gás (fica à esquerda); a chama sobe o calor.

## 4) Uma coleta para treinar, outra para testar

O teste que vale é numa coleta que o modelo **nunca viu**: outro momento, outro dia. Sortear
linhas para o teste dá nota alta à toa, porque um segundo e o seguinte são quase iguais (veja o
estudo de caso do Kaggle).

A regra desta célula: se o dataset tem mais de um dia, treina nos dias antigos e testa no dia
mais novo. Se tem um dia só, treina na primeira coleta e testa na última.

In [ ]:
coletas = sorted(df["coleta"].unique())
dias = sorted(df["coleta"].str[:10].unique())

TESTE = []
if len(dias) > 1:
    for c in coletas:
        if c.startswith(dias[-1]):
            TESTE.append(c)
else:
    TESTE.append(coletas[-1])

TREINO = []
for c in coletas:
    if c not in TESTE:
        TREINO.append(c)

treino = df[df["coleta"].isin(TREINO)]
teste = df[df["coleta"].isin(TESTE)]
print("treino:", TREINO, f"({len(treino)} linhas)")
print("teste: ", TESTE, f"({len(teste)} linhas)")

## 5) O modelo: uma árvore de decisão

Uma árvore com 3 níveis de perguntas. Dá para ler a regra que ela aprendeu.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text

FEATURES = ["temp", "umid", "raw_h2", "raw_ethanol", "pm2_5", "pm10", "nc0_5"]

arvore = DecisionTreeClassifier(max_depth=3, random_state=42)
arvore.fit(treino[FEATURES], treino["incendio"])

print(export_text(arvore, feature_names=FEATURES))

Leia de cima para baixo. Quais sensores a árvore combinou? Isso é sensor fusion: a regra
junta partícula, gás, calor e umidade.

## 6) O teste: a matriz de confusão

As linhas são a verdade (o botão); as colunas, o que o modelo disse.

- **Alarme falso:** era `not fire`, o modelo disse `fire`.
- **Fogo perdido:** era `fire`, o modelo disse `not fire`.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

pred = arvore.predict(teste[FEATURES])

matriz = ConfusionMatrixDisplay.from_predictions(teste["incendio"], pred, labels=["not fire", "fire"],
                                                 cmap="Blues", colorbar=False)
matriz.ax_.set_xlabel("o modelo disse")
matriz.ax_.set_ylabel("a verdade (o botão)")
plt.title("teste: " + ", ".join(TESTE))
plt.show()

sem_fogo = teste["incendio"] == "not fire"
com_fogo = teste["incendio"] == "fire"
alarme_falso = (pred[sem_fogo.values] == "fire").mean()
print(f"Alarme falso: {alarme_falso:.0%} dos segundos sem fogo")
if com_fogo.any():
    fogo_perdido = (pred[com_fogo.values] == "not fire").mean()
    print(f"Fogo perdido: {fogo_perdido:.0%} dos segundos com fogo")
else:
    print("Nesta coleta nao ha fogo: so da para medir o alarme falso.")

### Onde o modelo erra

De cada estímulo, quanto o modelo chamou de `fire` e de `not fire`.

In [ ]:
pd.crosstab(teste["estimulo"], pd.Series(pred, index=teste.index, name="o modelo disse"),
            normalize="index").round(2)

Num detector de incêndio, qual erro é pior: o alarme falso ou o fogo perdido? E o que acontece
com um detector que dispara alarme falso toda vez que alguém passa álcool nas mãos?

## 7) Usar o modelo: uma leitura nova

O modelo recebe **uma linha** com os 7 sensores e responde. Estas são as duas linhas do pico do
vapor e do pico da fumaça na coleta das 18:48: a partícula quase empata. O modelo acerta as
duas? Troque os números e teste outras leituras.

In [ ]:
leituras = pd.DataFrame([
    {"temp": 29.7, "umid": 47.7, "raw_h2": 14154, "raw_ethanol": 18910,
     "pm2_5": 155.4, "pm10": 183.0, "nc0_5": 806.6},    # pico do vapor
    {"temp": 29.0, "umid": 46.7, "raw_h2": 14172, "raw_ethanol": 18941,
     "pm2_5": 173.3, "pm10": 206.4, "nc0_5": 883.0},    # pico da fumaca
])

respostas = arvore.predict(leituras[FEATURES])
nomes = ["pico do vapor", "pico da fumaça"]
for nome, resposta in zip(nomes, respostas):
    print(f"{nome}: o modelo diz {resposta}")

## 8) O que levar

- **O CSV é o dataset:** cada segundo vira uma linha, e o rótulo veio do botão.
- **Sensor fusion:** um sensor sozinho engana; a árvore combina os sensores numa regra.
- **Teste numa coleta que o modelo nunca viu.** Sortear linhas dá nota alta à toa.
- **Dois erros, dois custos:** alarme falso e fogo perdido.